# 04 — Feature Engineering

## Project: Movie Recommendation System

### Objective

The objective of this notebook is to prepare movie information for a content-based recommendation system.

We will transform the cleaned movie dataset into a combined textual representation using relevant movie features.

### Main Tasks

1. Load the preprocessed movie dataset.
2. Inspect the available features.
3. Understand the features used for recommendation.
4. Combine genres, keywords, cast, director, and overview.
5. Normalize text values.
6. Create a combined tags feature.
7. Inspect the final feature representation.
8. Save the feature-engineered dataset.

### Expected Outcome

A dataset containing a combined textual feature for each movie, ready for vectorization in the next notebook.

In [20]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

## 3. Define Project Paths

We define the location of the preprocessed dataset and the directory where the feature-engineered dataset will be saved.

The preprocessed dataset is generated in Notebook 03.

In [21]:
PROJECT_ROOT = Path.cwd().parent

INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH = INTERIM_DATA_DIR / "movies_cleaned.csv"
OUTPUT_PATH = PROCESSED_DATA_DIR / "movies_features.csv"

assert INPUT_PATH.exists(), f"Dataset not found: {INPUT_PATH}"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully.")

Project paths configured successfully.


## 4. Load Preprocessed Dataset

Load the cleaned movie dataset generated in Notebook 03.

We will inspect its dimensions, columns, and sample records before applying feature engineering.

In [22]:
movies_df = pd.read_csv(INPUT_PATH)

print("Dataset shape:", movies_df.shape)

display(movies_df.head())

Dataset shape: (4803, 7)


,id,title,overview,genres,keywords,cast,director
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and prot...","['Action', 'Adventure', 'Fantasy', 'Science Fiction']","['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien pla...","['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']",['James Cameron']
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But...","['Adventure', 'Fantasy', 'Action']","['ocean', 'drug abuse', 'exotic island', 'east india trading company', ""love of one's life"", 'traitor', 'shipwreck', 'strong woman', 'ship', 'alli...","['Johnny Depp', 'Orlando Bloom', 'Keira Knightley']",['Gore Verbinski']
2,206647,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret se...,"['Action', 'Adventure', 'Crime']","['spy', 'based on novel', 'secret agent', 'sequel', 'mi6', 'british secret service', 'united kingdom']","['Daniel Craig', 'Christoph Waltz', 'Léa Seydoux']",['Sam Mendes']
3,49026,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation an...","['Action', 'Crime', 'Drama', 'Thriller']","['dc comics', 'crime fighter', 'terrorist', 'secret identity', 'burglar', 'hostage drama', 'time bomb', 'gotham city', 'vigilante', 'cover-up', 's...","['Christian Bale', 'Michael Caine', 'Gary Oldman']",['Christopher Nolan']
4,49529,John Carter,"John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic planet of Barsoom (Mars) and reluc...","['Action', 'Adventure', 'Science Fiction']","['based on novel', 'mars', 'medallion', 'space travel', 'princess', 'alien', 'steampunk', 'martian', 'escape', 'edgar rice burroughs', 'alien race...","['Taylor Kitsch', 'Lynn Collins', 'Samantha Morton']",['Andrew Stanton']


## 5. Inspect Available Features

Before combining features, we inspect the columns available in the dataset.

For content-based movie recommendation, we focus on information that describes the movie's content and characteristics.

Selected features:

- Genres
- Keywords
- Cast
- Director
- Overview

These features provide different types of information about each movie.

In [23]:
selected_features = ["genres", "keywords", "cast", "director", "overview"]

missing_columns = [
    column for column in selected_features if column not in movies_df.columns
]

if missing_columns:
    raise KeyError(f"Missing columns: {missing_columns}")

display(movies_df[selected_features].head())

,genres,keywords,cast,director,overview
0,"['Action', 'Adventure', 'Fantasy', 'Science Fiction']","['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien pla...","['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']",['James Cameron'],"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and prot..."
1,"['Adventure', 'Fantasy', 'Action']","['ocean', 'drug abuse', 'exotic island', 'east india trading company', ""love of one's life"", 'traitor', 'shipwreck', 'strong woman', 'ship', 'alli...","['Johnny Depp', 'Orlando Bloom', 'Keira Knightley']",['Gore Verbinski'],"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But..."
2,"['Action', 'Adventure', 'Crime']","['spy', 'based on novel', 'secret agent', 'sequel', 'mi6', 'british secret service', 'united kingdom']","['Daniel Craig', 'Christoph Waltz', 'Léa Seydoux']",['Sam Mendes'],A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret se...
3,"['Action', 'Crime', 'Drama', 'Thriller']","['dc comics', 'crime fighter', 'terrorist', 'secret identity', 'burglar', 'hostage drama', 'time bomb', 'gotham city', 'vigilante', 'cover-up', 's...","['Christian Bale', 'Michael Caine', 'Gary Oldman']",['Christopher Nolan'],"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation an..."
4,"['Action', 'Adventure', 'Science Fiction']","['based on novel', 'mars', 'medallion', 'space travel', 'princess', 'alien', 'steampunk', 'martian', 'escape', 'edgar rice burroughs', 'alien race...","['Taylor Kitsch', 'Lynn Collins', 'Samantha Morton']",['Andrew Stanton'],"John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic planet of Barsoom (Mars) and reluc..."


## 6. Understand Feature Representation

The genres, keywords, cast, and director columns were extracted in Notebook 03.

When saved as CSV, list values are stored as strings.

For example:

"['Action', 'Adventure', 'Science Fiction']"

Before combining these values, we convert them into consistent text representations.

The overview column is already textual.

In [24]:
for column in ["genres", "keywords", "cast", "director"]:
    print(f"\n{column}")
    print("Type:", type(movies_df[column].iloc[0]))
    print("Sample:", movies_df[column].iloc[0])

print("\nOverview")
print("Type:", type(movies_df["overview"].iloc[0]))
print("Sample:", movies_df["overview"].iloc[0])


genres
Type: <class 'str'>
Sample: ['Action', 'Adventure', 'Fantasy', 'Science Fiction']

keywords
Type: <class 'str'>
Sample: ['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']

cast
Type: <class 'str'>
Sample: ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

director
Type: <class 'str'>
Sample: ['James Cameron']

Overview
Type: <class 'str'>
Sample: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.


In [25]:
movies_df["genres"].iloc[0]

"['Action', 'Adventure', 'Fantasy', 'Science Fiction']"

## 7. Convert List-Like Strings into Lists

CSV files cannot directly preserve Python list objects.

Therefore, list values saved by Notebook 03 are loaded as strings.

We convert these string representations back into lists before combining their elements.

Invalid or missing values are handled safely.

In [26]:
# import ast


# def parse_list(value):
#     """Convert a string representation of a list into a Python list."""

#     if isinstance(value, list):
#         return value

#     if not isinstance(value, str) or not value.strip():
#         return []

#     try:
#         parsed = ast.literal_eval(value)

#         if isinstance(parsed, list):
#             return parsed

#     except (ValueError, SyntaxError):
#         pass

#     return []

In [ ]:
# import ast

# def parse_list(text):
#     # Check if the input is a valid string to prevent errors on NaN or re-runs
#     if not isinstance(text, str):
#         return ""

#     # Convert string to list of dicts and extract the 'name' values
#     items = ast.literal_eval(text)
#     return " ".join([i['name'] for i in items])


In [30]:
import ast


def parse_list(value):
    """
    Convert a serialized list into a Python list.

    Supports:
    - Lists of strings
    - Lists of dictionaries containing names
    - Missing or invalid values
    """

    if isinstance(value, list):
        items = value

    elif isinstance(value, str):
        if not value.strip():
            return []

        try:
            items = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []

    else:
        return []

    if not isinstance(items, list):
        return []

    result = []

    for item in items:
        if isinstance(item, str) and item.strip():
            result.append(item.strip())

        elif isinstance(item, dict):
            name = item.get("name")

            if isinstance(name, str) and name.strip():
                result.append(name.strip())

    return result

In [31]:
# movies_df['genres'] = movies_df['genres'].apply(lambda x :" ".join([i['name'] for i in ast.literal_eval(x)]))

In [32]:
list_columns = ["genres", "keywords", "cast", "director"]

for column in list_columns:
    movies_df[column] = movies_df[column].apply(parse_list)

display(movies_df[list_columns].head())

,genres,keywords,cast,director
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldi...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afte...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess,...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, alien race, superhuman strength...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


## 8. Normalize Text Features

Text normalization ensures that feature values follow a consistent format.

For this project, we:

- Convert text to lowercase.
- Remove unnecessary leading and trailing spaces.
- Convert list elements into normalized strings.

This helps reduce differences caused only by capitalization or extra whitespace.

In [33]:
def normalize_text(value):
    """Normalize a text value."""

    if not isinstance(value, str):
        return ""

    return " ".join(value.lower().split())


def normalize_list(values):
    """Normalize each string in a list."""

    return [
        normalize_text(value)
        for value in values
        if isinstance(value, str) and value.strip()
    ]

In [34]:
for column in list_columns:
    movies_df[column] = movies_df[column].apply(normalize_list)

movies_df["overview"] = movies_df["overview"].apply(normalize_text)

display(movies_df[selected_features].head())

,genres,keywords,cast,director,overview
0,"[action, adventure, fantasy, science fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldi...","[sam worthington, zoe saldana, sigourney weaver]",[james cameron],"in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and prot..."
1,"[adventure, fantasy, action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afte...","[johnny depp, orlando bloom, keira knightley]",[gore verbinski],"captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but..."
2,"[action, adventure, crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","[daniel craig, christoph waltz, léa seydoux]",[sam mendes],a cryptic message from bond’s past sends him on a trail to uncover a sinister organization. while m battles political forces to keep the secret se...
3,"[action, crime, drama, thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess,...","[christian bale, michael caine, gary oldman]",[christopher nolan],"following the death of district attorney harvey dent, batman assumes responsibility for dent's crimes to protect the late attorney's reputation an..."
4,"[action, adventure, science fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, alien race, superhuman strength...","[taylor kitsch, lynn collins, samantha morton]",[andrew stanton],"john carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic planet of barsoom (mars) and reluc..."


## 9. Combine Movie Features

A content-based recommendation system compares movies based on their characteristics.

Instead of processing each feature independently, we combine relevant movie information into a single textual representation.

The combined representation includes:

- Genres
- Keywords
- Cast
- Director
- Overview

This combined feature will be used to create numerical vectors in the next notebook.

In [35]:
def combine_features(row):
    """Combine selected movie features into one text representation."""

    genres = " ".join(row["genres"])
    keywords = " ".join(row["keywords"])
    cast = " ".join(row["cast"])
    director = " ".join(row["director"])
    overview = row["overview"]

    return " ".join(
        part for part in [genres, keywords, cast, director, overview] if part
    )

In [36]:
movies_df["tags"] = movies_df.apply(combine_features, axis=1)

display(movies_df[["title", "genres", "keywords", "cast", "director", "tags"]].head(5))

,title,genres,keywords,cast,director,tags
0,Avatar,"[action, adventure, fantasy, science fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldi...","[sam worthington, zoe saldana, sigourney weaver]",[james cameron],action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe ali...
1,Pirates of the Caribbean: At World's End,"[adventure, fantasy, action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afte...","[johnny depp, orlando bloom, keira knightley]",[gore verbinski],adventure fantasy action ocean drug abuse exotic island east india trading company love of one's life traitor shipwreck strong woman ship alliance...
2,Spectre,"[action, adventure, crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","[daniel craig, christoph waltz, léa seydoux]",[sam mendes],action adventure crime spy based on novel secret agent sequel mi6 british secret service united kingdom daniel craig christoph waltz léa seydoux s...
3,The Dark Knight Rises,"[action, crime, drama, thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess,...","[christian bale, michael caine, gary oldman]",[christopher nolan],action crime drama thriller dc comics crime fighter terrorist secret identity burglar hostage drama time bomb gotham city vigilante cover-up super...
4,John Carter,"[action, adventure, science fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, alien race, superhuman strength...","[taylor kitsch, lynn collins, samantha morton]",[andrew stanton],action adventure science fiction based on novel mars medallion space travel princess alien steampunk martian escape edgar rice burroughs alien rac...


## 10. Inspect Combined Features

We inspect the generated tags column to ensure that the selected movie features have been combined correctly.

The tags column should contain textual information describing each movie.

We also inspect the length of the combined text representation.

In [37]:
movies_df["tags_length"] = movies_df["tags"].str.len()

display(movies_df[["title", "tags", "tags_length"]].head(10))

,title,tags,tags_length
0,Avatar,action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe ali...,470
1,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drug abuse exotic island east india trading company love of one's life traitor shipwreck strong woman ship alliance...,446
2,Spectre,action adventure crime spy based on novel secret agent sequel mi6 british secret service united kingdom daniel craig christoph waltz léa seydoux s...,396
3,The Dark Knight Rises,action crime drama thriller dc comics crime fighter terrorist secret identity burglar hostage drama time bomb gotham city vigilante cover-up super...,742
4,John Carter,action adventure science fiction based on novel mars medallion space travel princess alien steampunk martian escape edgar rice burroughs alien rac...,619
5,Spider-Man 3,fantasy action adventure dual identity amnesia sandstorm love of one's life forgiveness spider wretch death of a friend egomania sand narcism host...,528
6,Tangled,animation family hostage magic horse fairy tale musical princess animation tower blonde woman selfishness healing power based on fairy tale during...,753
7,Avengers: Age of Ultron,action adventure science fiction marvel comic sequel superhero based on comic book vision superhero team duringcreditsstinger marvel cinematic uni...,608
8,Harry Potter and the Half-Blood Prince,adventure fantasy family witch magic broom school of witchcraft wizardry apparition teenage crush werewolf daniel radcliffe rupert grint emma wats...,335
9,Batman v Superman: Dawn of Justice,action adventure fantasy dc comics vigilante superhero based on comic book revenge super powers clark kent bruce wayne dc extended universe ben af...,562


In [38]:
print("Average tags length:", movies_df["tags_length"].mean())
print("Maximum tags length:", movies_df["tags_length"].max())
print("Minimum tags length:", movies_df["tags_length"].min())

print("Movies with empty tags:", (movies_df["tags"].str.strip() == "").sum())

Average tags length: 464.60691234645014
Maximum tags length: 1578
Minimum tags length: 23
Movies with empty tags: 0


## 11. Prepare Final Feature Dataset

We now select the columns required by the recommendation system.

The final dataset retains:

- id
- title
- tags

The ID uniquely identifies a movie, the title allows users to search for movies, and tags contain the combined textual representation used for recommendation.

In [39]:
movies_features = movies_df[["id", "title", "tags"]].copy()

movies_features = movies_features.dropna(subset=["id", "title", "tags"])

movies_features["tags"] = movies_features["tags"].astype(str)

display(movies_features.head())

,id,title,tags
0,19995,Avatar,action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe ali...
1,285,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drug abuse exotic island east india trading company love of one's life traitor shipwreck strong woman ship alliance...
2,206647,Spectre,action adventure crime spy based on novel secret agent sequel mi6 british secret service united kingdom daniel craig christoph waltz léa seydoux s...
3,49026,The Dark Knight Rises,action crime drama thriller dc comics crime fighter terrorist secret identity burglar hostage drama time bomb gotham city vigilante cover-up super...
4,49529,John Carter,action adventure science fiction based on novel mars medallion space travel princess alien steampunk martian escape edgar rice burroughs alien rac...


## 12. Save Feature-Engineered Dataset

Save the feature-engineered dataset in the data/processed directory.

This dataset will be used in Notebook 05 to create numerical vectors using TF-IDF or CountVectorizer, depending on the recommendation approach selected.

In [40]:
movies_features.to_csv(OUTPUT_PATH, index=False)

print(f"Feature dataset saved to: {OUTPUT_PATH}")
print(f"Total movies: {len(movies_features)}")

Feature dataset saved to: d:\Learning-Materials\Projects\ML-Projects\Content-Based-Movie-Recommendation-System\data\processed\movies_features.csv
Total movies: 4803


## 13. Conclusion

In this notebook, we performed feature engineering on the preprocessed movie dataset.

### Tasks Completed

- Loaded the cleaned movie dataset.
- Inspected the selected features.
- Converted list-like strings into lists.
- Normalized text values.
- Combined genres, keywords, cast, director, and overview.
- Created a combined tags feature.
- Inspected the generated text representations.
- Saved the feature-engineered dataset.

### Output

The final dataset is saved at:

data/processed/movies_features.csv

### Next Steps

In the next notebook, we will build the Content-Based Recommendation System.

We will transform the textual movie representations into numerical vectors and calculate similarity between movies.

**Next Notebook:** 05_content_based_recommendation.ipynb